# Neural-network round — how the networks fared

Reads completed runs from `mlflow.db`. **Fits nothing and imports no estimator**, so it
never loads torch alongside LightGBM (that combination segfaults on macOS — see
`src/home_credit_risk/evaluation.py`).

The round's headline: the best network reaches **0.78500** against the GBT's **0.79478**.
That gap is the expected result for heterogeneous tabular data, not a failure — but the
round's real value is the ablation evidence, where **three of four going-in predictions
turned out wrong**.

In [ ]:
import os
import sys
import warnings
from pathlib import Path

# Work from the repo root, not the notebook's directory. Three things depend on it:
# `config.Settings` discovers `.env` (and API_KEY has no default, so importing
# `tracking` fails without it), and both `sqlite:///mlflow.db` and `sqlite:///optuna.db`
# are relative URIs. Guarded so re-running the cell is a no-op.
HERE = [Path.cwd(), *Path.cwd().parents]
ROOT = next(p for p in HERE if (p / "pyproject.toml").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")

# Imports follow the path setup above, hence the E402 suppressions -- the same pattern
# the runners in scripts/ use.
import logging  # noqa: E402

import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import polars as pl  # noqa: E402

from home_credit_risk.tracking import fold_scores, run_history  # noqa: E402

# `home_credit_risk.logger` configures the ROOT logger, and APP_ENV=development sets it
# to DEBUG — which means matplotlib's font-matching chatter (hundreds of lines per
# figure) lands in every cell. Raise the floor for third-party loggers only; our own
# module's messages still come through.
for noisy in ("matplotlib", "PIL", "urllib3", "alembic", "mlflow", "asyncio"):
    logging.getLogger(noisy).setLevel(logging.WARNING)

plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

# The seed-noise floor measured across 5 NN seeds: any difference smaller than ~3 SE
# is not established by a single pair of runs. Drawn on every comparison below.
SEED_STD = 0.00030
SE_DIFF = SEED_STD * np.sqrt(2)
print(
    f"noise floor: std {SEED_STD:.5f}, SE of a difference {SE_DIFF:.5f}, "
    f"3 SE = {3 * SE_DIFF:.5f}"
)

## 1. Stage A — the harness had two bugs, and the sanity gate caught them

The plan set a gate before any result would be trusted: *the network must beat the
logistic challenger (0.77741). A network below a linear model on the same features is a
harness bug, not a modelling insight.*

It failed that gate — 0.77625 — and the gate was right. Two defects:

1. **OneCycleLR + early stopping are incompatible.** The best epoch was 10.8 of 60,
   landing exactly on the learning-rate peak, so training was truncated during the
   high-LR phase and the model never saw the annealing that produces its best weights.
2. **Imputation ran before scaling.** Filling nulls with the median first puts ~70% of
   the mass of a sparse column on a single value, and the rank transform then spreads
   that point mass across the output range as if it were signal.

In [ ]:
hist = run_history()
nn = hist.filter(pl.col("run_name").str.starts_with("nn-")).sort("started")

stage_a = ["nn-smoke", "nn-mlp-fullcycle", "nn-mlp-fixedprep", "nn-mlp-lowlr"]
a = nn.filter(pl.col("run_name").is_in(stage_a))
print(a.select(["run_name", "oof_roc_auc", "lr", "epochs", "patience"]))

LOGISTIC = 0.77741
fig, ax = plt.subplots(figsize=(8, 4))
order = [r for r in stage_a if r in a["run_name"].to_list()]
vals = [a.filter(pl.col("run_name") == r)["oof_roc_auc"][0] for r in order]
colors = ["#c53030" if v < LOGISTIC else "#2f855a" for v in vals]
ax.bar(range(len(order)), vals, color=colors)
ax.axhline(
    LOGISTIC, color="#c05621", ls="--", lw=1.5, label=f"logistic gate {LOGISTIC:.5f}"
)
ax.set_ylim(0.770, 0.785)
ax.set_xticks(range(len(order)))
ax.set_xticklabels(order, rotation=20, ha="right", fontsize=8)
ax.set_ylabel("OOF ROC-AUC")
ax.legend(fontsize=8)
ax.set_title("Stage A: below the gate (red) until the harness was fixed")
plt.tight_layout()

## 2. Stage B — the ablations, against the noise floor

Each run changes exactly one thing. The shaded band is **±3 SE of a difference between
two single-seed runs**; an effect inside it is not established by one pair of runs, no
matter which direction it points.

In [ ]:
# The Stage-B reference is the fixed-harness run: rank-gauss + embeddings + indicators.
# Every ablation below changes exactly one thing relative to it.
REF_B = "nn-mlp-lowlr"
ABLATIONS = {
    "nn-mlp-shortlist-200-stdscale": "standard scaling (vs rank-gauss)",
    "nn-mlp-shortlist-200-onehot": "one-hot (vs embeddings)",
    "nn-mlp-shortlist-200-noind": "drop missingness indicators",
    "nn-mlp-all-945": "all 945 features (vs 200)",
    "nn-mlp-stdscale-onehot": "both winners together",
}

ref_auc = nn.filter(pl.col("run_name") == REF_B)["oof_roc_auc"][0]
rows = []
for run, label in ABLATIONS.items():
    sub = nn.filter(pl.col("run_name") == run)
    if sub.height:
        d = sub["oof_roc_auc"][0] - ref_auc
        rows.append(
            {
                "change": label,
                "oof": sub["oof_roc_auc"][0],
                "delta": d,
                "SE": abs(d) / SE_DIFF,
                "verdict": "established"
                if abs(d) > 3 * SE_DIFF
                else "inside noise floor",
            }
        )
abl = pl.DataFrame(rows).sort("delta")
print(f"reference: {REF_B} = {ref_auc:.5f}\n")
print(abl)

# The two winners combined gain less than the sum of their parts -- checked by running
# it rather than assumed.
combined = abl.filter(pl.col("change") == "both winners together")
parts = abl.filter(pl.col("change").is_in(list(ABLATIONS.values())[:2]))["delta"].sum()
if combined.height:
    print(
        f"\nsub-additive: {parts:+.5f} predicted by addition, "
        f"{combined['delta'][0]:+.5f} measured"
    )

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
y = np.arange(abl.height)
d = abl["delta"].to_numpy()
colors = ["#2f855a" if v > 0 else "#c53030" for v in d]
colors = [
    c if v == "established" else "#a0aec0" for c, v in zip(colors, abl["verdict"])
]
ax.barh(y, d, color=colors)
ax.axvspan(
    -3 * SE_DIFF,
    3 * SE_DIFF,
    color="#718096",
    alpha=0.18,
    label="±3 SE (not established)",
)
ax.axvline(0, color="black", lw=1)
ax.set_yticks(y)
ax.set_yticklabels(abl["change"], fontsize=9)
ax.set_xlabel("Δ OOF ROC-AUC vs reference")
ax.set_title(f"Stage B ablations vs {REF_B} — grey bars are inside the noise floor")
ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()

### Three of four predictions were wrong

| Prediction | Outcome |
|---|---|
| Rank-gauss beats standard scaling — 53 features have `max/p99 > 10`, up to 2810× | **Wrong.** Standard scaling won. The rank transform discards magnitude, and BatchNorm re-normalises the tails anyway. |
| Entity embeddings beat one-hot | **Wrong.** The widest categorical has 58 levels, where a 13-dim embedding is a bottleneck, not a compression. |
| ResNet matches or beats the plain MLP | **Wrong.** It lost by 0.0034. |
| Missingness indicators help | **Right** (+0.00117) — but at 2.8 SE, still short of the 3 SE bar. |

The two effects nearest the floor (indicators 2.8 SE, one-hot 2.2 SE) are recorded as
**directional only**. They set the defaults because a decision had to be made, not
because they were proven.

## 3. Stage C — what actually moved the number

Ensembling five seeds and rank-averaging beat every preprocessing and architecture
change in the round, at zero design risk. Rank-averaging (not probability averaging)
because AUC depends only on ordering, so it is robust to seeds whose probability scales
differ.

In [ ]:
# Stage C compares against the *settled* preprocessing, not the Stage-B reference:
# architectures are only comparable once preprocessing is frozen.
REF_C = "nn-mlp-stdscale-onehot"
single = nn.filter(pl.col("run_name") == REF_C)["oof_roc_auc"][0]
print(f"reference (MLP, settled preprocessing)  {single:.5f}\n")

for run, label in [
    ("nn-resnet-shortlist-200", "ResNet-MLP"),
    ("nn-mlp-shortlist-200-x5", "MLP, 5-seed rank-average"),
]:
    sub = nn.filter(pl.col("run_name") == run)
    if sub.height:
        d = sub["oof_roc_auc"][0] - single
        mark = "established" if abs(d) > 3 * SE_DIFF else "inside noise floor"
        print(
            f"{label:26s} {sub['oof_roc_auc'][0]:.5f}   {d:+.5f}"
            f"   {abs(d) / SE_DIFF:4.1f} SE  ({mark})"
        )

print("\nThe ensemble is the largest established effect in the round — larger than")
print("any preprocessing or architecture change, at no design risk.")

## 4. Cross-family comparison, paired by fold

All three families share the split, so this is a paired comparison. Note **fold 2 is the
easiest fold for every model and fold 3 among the hardest** — the folds differ in
difficulty far more than the models differ from each other, which is exactly why an
unpaired comparison of OOF scalars would be the weaker test.

In [ ]:
FAMILIES = {
    "gbt-final": "GBT (945 feat)",
    "nn-mlp-shortlist-200-x5": "NN 5-seed ensemble",
    "logistic-shortlist": "Logistic",
}
fs = fold_scores(list(FAMILIES))
if fs.height:
    wide = fs.pivot(values="score", index="fold", on="run_name").sort("fold")
    print(wide)

    fig, ax = plt.subplots(figsize=(8, 4.5))
    for run, label in FAMILIES.items():
        if run in wide.columns:
            ax.plot(wide["fold"], wide[run], "o-", lw=2, label=label)
    ax.set_xticks(wide["fold"])
    ax.set_xlabel("fold")
    ax.set_ylabel("ROC-AUC")
    ax.set_title("Same folds, three model families")
    ax.legend(fontsize=8)
    plt.tight_layout()

## 5. Leaderboard

| Model | Features | OOF ROC-AUC |
|---|---|---|
| **LightGBM GBT** | 945 | **0.79478** |
| LightGBM GBT | 200 | 0.79456 |
| NN, 5-seed ensemble | 200 | 0.78500 |
| NN, single seed | 200 | 0.78235 |
| ResNet-MLP | 200 | 0.77900 |
| Logistic regression | 200 | 0.77741 |

**tree > network > linear** is the expected ordering for heterogeneous tabular data.

One asymmetry worth keeping: the 200-feature shortlist — produced *by* the tree — is
worth **+0.0027 to the network and −0.0002 to the tree itself**. The tree does its own
feature selection internally and gains nothing from being handed the answer; the network
has no such mechanism and benefits from the tree's.

### What this notebook cannot show

ROC curves, calibration plots and decile lift need the **out-of-fold prediction vectors**,
and only summary scalars were logged — `CVResult.oof_pred` was never persisted. Adding
that to `log_cv_metrics` and re-running the two headline configs would enable them.